
    NVIDIA GPU SALES ETL

In [60]:
#Load libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mysql.connector
from sqlalchemy import create_engine



In [61]:
#Load Dataset

df = pd.read_csv("../data/raw/nvidia_gpu_sales_synthetic_2026.csv")


In [62]:
#Exploring Dataset-->
df.head()

,sale_id,sale_date,gpu_model,gpu_family,launch_year,region,sales_channel,customer_segment,units_sold,msrp_usd,avg_street_price_usd,price_premium_pct,stock_status,customer_satisfaction_score,warranty_months,bundle_addon,revenue_usd
0,1,2024-01-12,RTX 4090,Consumer Gaming,2022,Asia-Pacific (ex-China),Retail/Etail,Gaming,8,1599,1796.91,12.38,Low Stock,4.0,12,Software Bundle,14375.30
1,2,2024-01-18,RTX 4070 Ti Super,Consumer Gaming,2024,Asia-Pacific (ex-China),System Integrator/OEM,Content Creation,34,799,964.70,20.74,Low Stock,5.0,12,NaN,32799.82
2,3,2024-01-20,RTX 4080 Super,Consumer Gaming,2024,North America,Retail/Etail,Gaming,30,999,1308.25,30.96,Sold Out,4.0,12,NaN,39247.64
3,4,2024-02-05,H100 SXM,Data Center AI,2022,Asia-Pacific (ex-China),Direct Enterprise,Crypto Mining,5,27000,29202.11,8.16,Low Stock,4.5,24,NaN,146010.54
4,5,2024-02-16,H100 SXM,Data Center AI,2022,Europe,Cloud Provider,Crypto Mining,9,27000,27778.73,2.88,In Stock,4.5,36,NaN,250008.56


In [63]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")       #-----> Check shape of dataset (rows & columns)


Rows: 7000
Columns: 17


In [64]:
df.columns        #-----> Check column names

Index(['sale_id', 'sale_date', 'gpu_model', 'gpu_family', 'launch_year',
       'region', 'sales_channel', 'customer_segment', 'units_sold', 'msrp_usd',
       'avg_street_price_usd', 'price_premium_pct', 'stock_status',
       'customer_satisfaction_score', 'warranty_months', 'bundle_addon',
       'revenue_usd'],
      dtype='str')

In [65]:
df.info()        #-----> show information of dataset

<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   sale_id                      7000 non-null   int64  
 1   sale_date                    7000 non-null   str    
 2   gpu_model                    7000 non-null   str    
 3   gpu_family                   7000 non-null   str    
 4   launch_year                  7000 non-null   int64  
 5   region                       7000 non-null   str    
 6   sales_channel                7000 non-null   str    
 7   customer_segment             7000 non-null   str    
 8   units_sold                   7000 non-null   int64  
 9   msrp_usd                     7000 non-null   int64  
 10  avg_street_price_usd         7000 non-null   float64
 11  price_premium_pct            7000 non-null   float64
 12  stock_status                 7000 non-null   str    
 13  customer_satisfaction_score  

In [66]:
df.describe()        #---->  Statistical description of dataset

,sale_id,launch_year,units_sold,msrp_usd,avg_street_price_usd,price_premium_pct,customer_satisfaction_score,warranty_months,revenue_usd
count,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7.000000e+03
mean,3500.500000,2024.264429,21.674143,5212.903857,6042.141264,15.010536,4.428286,19.753714,5.658006e+04
std,2020.870275,1.100049,12.928136,11056.780783,12976.951044,14.348016,0.608437,8.637874,9.996269e+04
min,1.000000,2020.000000,1.000000,549.000000,549.050000,0.000000,1.500000,12.000000,1.393130e+03
25%,1750.750000,2024.000000,12.000000,749.000000,805.090000,4.560000,4.000000,12.000000,1.555133e+04
50%,3500.500000,2025.000000,20.000000,999.000000,1048.015000,8.850000,4.500000,24.000000,2.534293e+04
75%,5250.250000,2025.000000,29.000000,1599.000000,2002.570000,21.352500,5.000000,24.000000,4.670657e+04
max,7000.000000,2025.000000,104.000000,42000.000000,82364.480000,98.460000,5.000000,36.000000,1.225726e+06


In [67]:
df.duplicated().sum()    #-----> check duplicate data

np.int64(0)

In [68]:
df.dtypes    #-----> check datatypes 


sale_id                          int64
sale_date                          str
gpu_model                          str
gpu_family                         str
launch_year                      int64
region                             str
sales_channel                      str
customer_segment                   str
units_sold                       int64
msrp_usd                         int64
avg_street_price_usd           float64
price_premium_pct              float64
stock_status                       str
customer_satisfaction_score    float64
warranty_months                  int64
bundle_addon                       str
revenue_usd                    float64
dtype: object

In [69]:
df["sale_date"] = pd.to_datetime(df["sale_date"])        # change datatype of sales_date column

In [70]:
# convert categorial columns

categorical_columns = [
    "gpu_model",
    "gpu_family",
    "region",
    "sales_channel",
    "customer_segment",
    "stock_status",
    "bundle_addon"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")

In [71]:
# convert numerical columns 

numeric_columns = [
    "launch_year",
    "units_sold",
    "msrp_usd",
    "avg_street_price_usd",
    "price_premium_pct",
    "customer_satisfaction_score",
    "warranty_months",
    "revenue_usd"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [72]:
df.dtypes

sale_id                                 int64
sale_date                      datetime64[us]
gpu_model                            category
gpu_family                           category
launch_year                             int64
region                               category
sales_channel                        category
customer_segment                     category
units_sold                              int64
msrp_usd                                int64
avg_street_price_usd                  float64
price_premium_pct                     float64
stock_status                         category
customer_satisfaction_score           float64
warranty_months                         int64
bundle_addon                         category
revenue_usd                           float64
dtype: object

Check Invalid Values

In [73]:
# Units sold can not be negative

df[df["units_sold"] < 0]       #-----> check if column contains values less than zero

,sale_id,sale_date,gpu_model,gpu_family,launch_year,region,sales_channel,customer_segment,units_sold,msrp_usd,avg_street_price_usd,price_premium_pct,stock_status,customer_satisfaction_score,warranty_months,bundle_addon,revenue_usd


In [74]:
#Revenue shuld be positive 

df = df[df["revenue_usd"] >= 0]

In [75]:
#Customer satisfaction score shuld be in 0 to 10

df = df[
    (df["customer_satisfaction_score"] >= 0) &
    (df["customer_satisfaction_score"] <= 10)
]

In [76]:
#Handel missing values

df.isnull().sum()   #-----> Checking Missing Values


sale_id                           0
sale_date                         0
gpu_model                         0
gpu_family                        0
launch_year                       0
region                            0
sales_channel                     0
customer_segment                  0
units_sold                        0
msrp_usd                          0
avg_street_price_usd              0
price_premium_pct                 0
stock_status                      0
customer_satisfaction_score       0
warranty_months                   0
bundle_addon                   3841
revenue_usd                       0
dtype: int64

In [77]:
df["bundle_addon"] = df["bundle_addon"].astype("object")

df["bundle_addon"] = df["bundle_addon"].fillna("No Bundle")       #----> filling null values

df["bundle_addon"] = df["bundle_addon"].astype("category")

Feature Engineering - Creating Usefull features

In [78]:
#creating estimated profit column

df["estimated_profit"] = (
    df["revenue_usd"] -
    (df["units_sold"] * df["avg_street_price_usd"])
)

In [79]:
# creating revenue per unit column

df["revenue_per_unit"] = (
    df["revenue_usd"] /
    df["units_sold"]
)

In [80]:
#creating premium category column 

df["premium_gpu"] = np.where(
    df["price_premium_pct"] > 20,
    "High Premium",
    "Normal"
)

In [55]:
#Remove useless features

df.drop("sale_id", axis=1, inplace=True)

Final cheack

In [ ]:
df.head()

,sale_date,gpu_model,gpu_family,launch_year,region,sales_channel,customer_segment,units_sold,msrp_usd,avg_street_price_usd,price_premium_pct,stock_status,customer_satisfaction_score,warranty_months,bundle_addon,revenue_usd,estimated_profit,revenue_per_unit,premium_gpu
0,2024-01-12,RTX 4090,Consumer Gaming,2022,Asia-Pacific (ex-China),Retail/Etail,Gaming,8,1599,1796.91,12.38,Low Stock,4.0,12,Software Bundle,14375.30,0.02,1796.912500,Normal
1,2024-01-18,RTX 4070 Ti Super,Consumer Gaming,2024,Asia-Pacific (ex-China),System Integrator/OEM,Content Creation,34,799,964.70,20.74,Low Stock,5.0,12,No Bundle,32799.82,0.02,964.700588,High Premium
2,2024-01-20,RTX 4080 Super,Consumer Gaming,2024,North America,Retail/Etail,Gaming,30,999,1308.25,30.96,Sold Out,4.0,12,No Bundle,39247.64,0.14,1308.254667,High Premium
3,2024-02-05,H100 SXM,Data Center AI,2022,Asia-Pacific (ex-China),Direct Enterprise,Crypto Mining,5,27000,29202.11,8.16,Low Stock,4.5,24,No Bundle,146010.54,-0.01,29202.108000,Normal
4,2024-02-16,H100 SXM,Data Center AI,2022,Europe,Cloud Provider,Crypto Mining,9,27000,27778.73,2.88,In Stock,4.5,36,No Bundle,250008.56,-0.01,27778.728889,Normal


In [57]:
df.describe()

,sale_date,launch_year,units_sold,msrp_usd,avg_street_price_usd,price_premium_pct,customer_satisfaction_score,warranty_months,revenue_usd,estimated_profit,revenue_per_unit
count,7000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000,7.000000e+03,7000.000000,7000.000000
mean,2025-10-13 04:41:37.371428,2024.264429,21.674143,5212.903857,6042.141264,15.010536,4.428286,19.753714,5.658006e+04,0.000589,6042.141279
min,2024-01-12 00:00:00,2020.000000,1.000000,549.000000,549.050000,0.000000,1.500000,12.000000,1.393130e+03,-0.420000,549.046765
25%,2025-07-01 00:00:00,2024.000000,12.000000,749.000000,805.090000,4.560000,4.000000,12.000000,1.555133e+04,-0.040000,805.092620
50%,2025-11-13 00:00:00,2025.000000,20.000000,999.000000,1048.015000,8.850000,4.500000,24.000000,2.534293e+04,0.000000,1048.015000
75%,2026-03-01 00:00:00,2025.000000,29.000000,1599.000000,2002.570000,21.352500,5.000000,24.000000,4.670657e+04,0.040000,2002.571501
max,2026-06-29 00:00:00,2025.000000,104.000000,42000.000000,82364.480000,98.460000,5.000000,36.000000,1.225726e+06,0.420000,82364.484000
std,NaN,1.100049,12.928136,11056.780783,12976.951044,14.348016,0.608437,8.637874,9.996269e+04,0.073024,12976.951039


In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   sale_date                    7000 non-null   datetime64[us]
 1   gpu_model                    7000 non-null   category      
 2   gpu_family                   7000 non-null   category      
 3   launch_year                  7000 non-null   int64         
 4   region                       7000 non-null   category      
 5   sales_channel                7000 non-null   category      
 6   customer_segment             7000 non-null   category      
 7   units_sold                   7000 non-null   int64         
 8   msrp_usd                     7000 non-null   int64         
 9   avg_street_price_usd         7000 non-null   float64       
 10  price_premium_pct            7000 non-null   float64       
 11  stock_status                 7000 non-null   category 

save final cleaned dataset

In [81]:
df.to_csv(
    "../data/processed/nvidia_gpu_sales_cleaned.csv",
    index=False
)